# UD2/UD3 - EDA, Preparación y Modelado

Cuaderno guía con dos casos clásicos:
- **House Prices** (regresión).
- **Titanic** (clasificación).

Secuencia alineada con los pasos de EDA, preparación y modelado de los documentos de la carpeta `Documentacion`. Incluye celdas con código comentado para que el alumnado tome decisiones (marcadas como TODO).

## Dependencias y configuración
- Propósito: cargar librerías principales y configurar rutas base.
- Uso: ajusta las rutas si los CSV están en otra carpeta.
- Nota: se asume que los datasets se han descargado previamente (Kaggle).

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_log_error,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor

sns.set_theme(style="whitegrid")
DATA_DIR = Path("data")
HOUSE_DIR = DATA_DIR / "house_prices"
TITANIC_DIR = DATA_DIR / "titanic"

## Utilidades comunes
Funciones de apoyo para resumen de faltantes y evaluación de modelos.

In [ ]:
def missing_report(df, top=20):
    miss = df.isna().mean().sort_values(ascending=False)
    return miss[miss > 0].head(top)

def eval_regression(y_true, y_pred, label="modelo"):
    metrics = {
        "rmse": mean_squared_error(y_true, y_pred, squared=False),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }
    print(label, metrics)
    return metrics

def eval_classification(y_true, y_pred, y_proba=None, label="modelo"):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_proba)
        except ValueError:
            pass
    print(label, metrics)
    return metrics

---
# Parte A: House Prices (Regresión)

### Carga de datos (House Prices)
- Propósito: cargar train/test de Kaggle (House Prices - Advanced Regression Techniques).
- Descarga (si hace falta): usa la API de Kaggle (`kaggle competitions download -c house-prices-advanced-regression-techniques`).
- Ajusta las rutas si ya tienes los CSV en otro lugar.

In [ ]:
train_hp_path = HOUSE_DIR / "train.csv"
test_hp_path = HOUSE_DIR / "test.csv"

if not train_hp_path.exists():
    raise FileNotFoundError(f"No se encontró {train_hp_path}. Descarga con Kaggle API y descomprime.")

hp_train = pd.read_csv(train_hp_path)
hp_test = pd.read_csv(test_hp_path) if test_hp_path.exists() else None
hp_train.shape, hp_test.shape if hp_test is not None else None

### EDA rápido: estructura y tipos
- Verifica dimensiones, primeras filas, tipos y uso de memoria.

In [ ]:
hp_train.head()

In [ ]:
hp_train.info()

In [ ]:
hp_train.describe().T.head()

### EDA: faltantes y variables
- Resumen de NaNs (top 20).
- Separación de numéricas/categóricas para guiar la preparación.

In [ ]:
hp_missing = missing_report(hp_train, top=30)
hp_missing

In [ ]:
num_cols_hp = hp_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_hp = [c for c in hp_train.columns if c not in num_cols_hp]
num_cols_hp[:10], cat_cols_hp[:10]

### EDA visual (opcional, ejecutar según tiempo)

In [ ]:
# Histograma rápido de la variable objetivo
sns.histplot(hp_train['SalePrice'], kde=True)
plt.title('Distribución de SalePrice')
plt.tight_layout()

In [ ]:
# TODO: Añade más visualizaciones univariantes/bivariantes relevantes (ej.: correlación con OverallQual)
# sns.boxplot(data=hp_train, x='OverallQual', y='SalePrice')

### Decisiones sobre faltantes (House Prices)
- Ejemplo: eliminar columnas con >40% de faltantes.
- Imputar numéricas con mediana y categóricas con moda.
- Deja una celda TODO para que el alumnado decida otras reglas (ej. imputar por grupo).

In [ ]:
missing_threshold = 0.4
cols_drop_hp = hp_missing[hp_missing > missing_threshold].index.tolist()

hp_train_clean = hp_train.drop(columns=cols_drop_hp)
if hp_test is not None:
    hp_test_clean = hp_test.drop(columns=cols_drop_hp)

num_cols_hp = hp_train_clean.select_dtypes(include=[np.number]).columns.tolist()
num_cols_hp.remove('SalePrice')  # objetivo
cat_cols_hp = [c for c in hp_train_clean.columns if c not in num_cols_hp + ['SalePrice']]
cols_drop_hp

In [ ]:
# TODO: Decide imputaciones específicas (ej. LotFrontage por vecindario, GarageYrBlt por GarageType)
# hp_train_clean['LotFrontage'] = hp_train_clean.groupby('Neighborhood')['LotFrontage'].transform(lambda s: s.fillna(s.median()))

### Preparación: transformaciones y pipelines (House Prices)
- Objetivo: dejar `X_train`, `X_val`, `y_train`, `y_val` listos con pipelines de preprocesado.
- Se usa ColumnTransformer con imputación + one-hot + escalado.

In [ ]:
X_hp = hp_train_clean.drop(columns=['SalePrice'])
y_hp = hp_train_clean['SalePrice']

X_train_hp, X_val_hp, y_train_hp, y_val_hp = train_test_split(
    X_hp, y_hp, test_size=0.2, random_state=42
)

numeric_pipe_hp = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe_hp = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocess_hp = ColumnTransformer([
    ("num", numeric_pipe_hp, num_cols_hp),
    ("cat", categorical_pipe_hp, cat_cols_hp),
])

### Modelos base (House Prices)
- Regresión lineal y Gradient Boosting.
- Métricas: RMSE, MAE, R².

In [ ]:
models_hp = {
    "LinearRegression": LinearRegression(),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

results_hp = {}
for name, model in models_hp.items():
    pipe = Pipeline([("prep", preprocess_hp), ("model", model)])
    pipe.fit(X_train_hp, y_train_hp)
    pred = pipe.predict(X_val_hp)
    results_hp[name] = eval_regression(y_val_hp, pred, label=name)
results_hp

### Optimización de hiperparámetros (House Prices)
- Ejemplo con RandomForestRegressor y búsqueda en rejilla ligera.
- Ajusta el espacio de búsqueda según recursos.

In [ ]:
rf_hp = RandomForestRegressor(random_state=42, n_estimators=200)
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2, 4],
}

rf_pipe = Pipeline([("prep", preprocess_hp), ("model", rf_hp)])

grid_hp = GridSearchCV(rf_pipe, param_grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1)
# grid_hp.fit(X_hp, y_hp)  # Descomenta para entrenar (coste alto)
# grid_hp.best_params_, -grid_hp.best_score_

---
# Parte B: Titanic (Clasificación)

### Carga de datos (Titanic)
- Propósito: cargar train/test del reto Titanic (Kaggle).
- Descarga: `kaggle competitions download -c titanic`.

In [ ]:
train_ti_path = TITANIC_DIR / "train.csv"
test_ti_path = TITANIC_DIR / "test.csv"

if not train_ti_path.exists():
    raise FileNotFoundError(f"No se encontró {train_ti_path}. Descarga con Kaggle API y descomprime.")

ti_train = pd.read_csv(train_ti_path)
ti_test = pd.read_csv(test_ti_path) if test_ti_path.exists() else None
ti_train.shape, ti_test.shape if ti_test is not None else None

### EDA rápido (Titanic)

In [ ]:
ti_train.head()

In [ ]:
ti_train.info()

In [ ]:
ti_missing = missing_report(ti_train)
ti_missing

### EDA visual (Titanic)

In [ ]:
# Distribución de la variable objetivo
sns.countplot(data=ti_train, x='Survived')
plt.title('Balance de clases')
plt.tight_layout()

In [ ]:
# TODO: Añade gráficos bivariantes (ej.: Survived vs Sex, Survived vs Pclass)
# sns.barplot(data=ti_train, x='Sex', y='Survived')

### Manejo de faltantes (Titanic)
- Estrategia ejemplo: eliminar `Cabin` por alta tasa de NaNs y completar `Age`/`Fare` con mediana, `Embarked` con moda.
- Celda TODO para definir otras imputaciones (p. ej. `Age` por título).

In [ ]:
cols_drop_ti = ['Cabin']
ti_train_clean = ti_train.drop(columns=[c for c in cols_drop_ti if c in ti_train.columns])
if ti_test is not None:
    ti_test_clean = ti_test.drop(columns=[c for c in cols_drop_ti if c in ti_test.columns])

num_cols_ti = ti_train_clean.select_dtypes(include=[np.number]).columns.tolist()
num_cols_ti.remove('Survived')
cat_cols_ti = [c for c in ti_train_clean.columns if c not in num_cols_ti + ['Survived']]

ti_train_clean[num_cols_ti] = ti_train_clean[num_cols_ti].fillna(ti_train_clean[num_cols_ti].median())
ti_train_clean[cat_cols_ti] = ti_train_clean[cat_cols_ti].fillna(ti_train_clean[cat_cols_ti].mode().iloc[0])

In [ ]:
# TODO: Imputación alternativa de Age por título extraído de Name
# ti_train_clean['Title'] = ti_train_clean['Name'].str.extract(' ([A-Za-z]+)\.')
# title_medians = ti_train_clean.groupby('Title')['Age'].transform('median')
# ti_train_clean['Age'] = ti_train_clean['Age'].fillna(title_medians)

### Preparación y split (Titanic)

In [ ]:
X_ti = ti_train_clean.drop(columns=['Survived'])
y_ti = ti_train_clean['Survived']

X_train_ti, X_val_ti, y_train_ti, y_val_ti = train_test_split(
    X_ti, y_ti, test_size=0.2, random_state=42, stratify=y_ti
)

numeric_pipe_ti = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe_ti = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocess_ti = ColumnTransformer([
    ("num", numeric_pipe_ti, num_cols_ti),
    ("cat", categorical_pipe_ti, cat_cols_ti),
])

### Modelos base (Titanic)
- Regresión logística y Random Forest.
- Métricas: accuracy, precision, recall, f1, roc_auc.

In [ ]:
models_ti = {
    "LogReg": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(random_state=42, n_estimators=300),
}

results_ti = {}
for name, model in models_ti.items():
    pipe = Pipeline([("prep", preprocess_ti), ("model", model)])
    pipe.fit(X_train_ti, y_train_ti)
    pred = pipe.predict(X_val_ti)
    proba = pipe.predict_proba(X_val_ti)[:, 1] if hasattr(pipe, "predict_proba") else None
    results_ti[name] = eval_classification(y_val_ti, pred, proba, label=name)
results_ti

### Optimización de hiperparámetros (Titanic)
- Ejemplo de GridSearch para RandomForestClassifier (usa CV estratificada).

In [ ]:
rf_ti = RandomForestClassifier(random_state=42)
param_grid_ti = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 8, 12],
    "model__min_samples_leaf": [1, 2, 4],
}

rf_pipe_ti = Pipeline([("prep", preprocess_ti), ("model", rf_ti)])

grid_ti = GridSearchCV(rf_pipe_ti, param_grid_ti, cv=5, scoring="roc_auc", n_jobs=-1)
# grid_ti.fit(X_ti, y_ti)
# grid_ti.best_params_, grid_ti.best_score_

### Evaluación adicional (Titanic)

In [ ]:
# TODO: Tras entrenar el mejor modelo, dibuja curva ROC y matriz de confusión
# RocCurveDisplay.from_estimator(grid_ti.best_estimator_, X_val_ti, y_val_ti)
# ConfusionMatrixDisplay.from_estimator(grid_ti.best_estimator_, X_val_ti, y_val_ti, normalize='true')
# plt.show()

## Exportar datasets limpios (opcional)

In [ ]:
# Guardar versiones limpias para uso posterior
# hp_train_clean.to_csv(DATA_DIR / 'house_prices_train_clean.csv', index=False)
# ti_train_clean.to_csv(DATA_DIR / 'titanic_train_clean.csv', index=False)

## Notas finales
- Sigue el flujo: EDA → decisiones de limpieza → preparación → modelado → ajuste de hiperparámetros.
- Las celdas TODO están comentadas para que el alumnado tome decisiones y las active.
- Si el entorno tiene GPU/CUDA (Colab/Kaggle), se pueden sustituir modelos por versiones de cuML o XGBoost.